In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/artigo2-xai'
for d in ['data/raw','data/splits','notebooks','models','results/metrics','results/confusion_matrix','results/gradcam']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
print(sorted(os.listdir(BASE)))

In [ ]:
!pip -q install grad-cam h5py

In [ ]:
import random, numpy as np, torch
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()
print('Seed fixada:', SEED)

In [ ]:
import sys, torch, numpy as np, torchvision, sklearn
print(sys.version)
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('numpy', np.__version__, '| sklearn', sklearn.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
BASE = '/content/drive/MyDrive/xai-brain-tumor'
print('Drive montado?', os.path.exists('/content/drive/MyDrive'))
print('Pasta do projeto existe?', os.path.exists(BASE))
if os.path.exists('/content/drive/MyDrive'):
    print('Conteúdo do Drive:', os.listdir('/content/drive/MyDrive')[:15])

In [ ]:
  for d in ['data/raw','data/splits','notebooks','models','results/metrics','results/confusion_matrix','results/gradcam']:
      os.makedirs(f'{BASE}/{d}', exist_ok=True)
  print(sorted(os.listdir(BASE)))

In [ ]:
BASE = '/content/drive/MyDrive/xai-brain-tumor'
!pip freeze | grep -iE "^(torch|torchvision|numpy|pandas|scipy|h5py|scikit-learn|matplotlib|seaborn|pillow|opencv-python|tqdm|grad-cam)==" > {BASE}/requirements.txt
!cat {BASE}/requirements.txt

In [ ]:
import os
BASE = '/content/drive/MyDrive/xai-brain-tumor'
RAW = f'{BASE}/data/raw'
for f in sorted(os.listdir(RAW)):
    print(f, round(os.path.getsize(f'{RAW}/{f}')/1e6, 1), 'MB')

In [ ]:
fonte = """Dataset: brain tumor dataset (Jun Cheng), Figshare
https://doi.org/10.6084/m9.figshare.1512427
Versao baixada: Version 8 (postada em 2024-12-21)
Artigo: Cheng et al., Enhanced Performance of Brain Tumor Classification via Tumor Region Augmentation and Partition, PLoS ONE 10(10), 2015
"""
open(f'{RAW}/FONTE.txt', 'w').write(fonte)
print('FONTE.txt salvo')

In [ ]:
import glob, os
OUT = '/content/figshare'
os.makedirs(OUT, exist_ok=True)
os.makedirs('/content/brain_tumor_mat', exist_ok=True)

outer = sorted(glob.glob(f'{RAW}/*.zip'))
print('zips em raw:', [os.path.basename(z) for z in outer])
for z in outer:
    !unzip -q -o "{z}" -d {OUT}

for z in sorted(glob.glob(f'{OUT}/**/*.zip', recursive=True)):
    print('abrindo', os.path.basename(z))
    !unzip -q -o "{z}" -d /content/brain_tumor_mat

files = sorted(f for f in glob.glob('/content/**/*.mat', recursive=True)
               if os.path.basename(f)[:-4].isdigit() and 'brain_tumor_mat' in f or
                  os.path.basename(f)[:-4].isdigit() and 'figshare' in f)
print(len(files), 'arquivos .mat')
print(files[:2])
print(sorted(os.listdir(OUT)))

In [ ]:
import h5py, numpy as np

def load_case(path):
    try:
        with h5py.File(path, 'r') as f:
            d = f['cjdata']
            img = np.array(d['image']).T.astype(np.float32)
            mask = np.array(d['tumorMask']).T.astype(bool)
            label = int(np.array(d['label']).flatten()[0])
            pid = ''.join(chr(int(c)) for c in np.array(d['PID']).flatten())
    except OSError:
        from scipy.io import loadmat
        d = loadmat(path, struct_as_record=False, squeeze_me=True)['cjdata']
        img = np.array(d.image, dtype=np.float32)
        mask = np.array(d.tumorMask).astype(bool)
        label = int(d.label)
        pid = str(d.PID)
    return img, mask, label - 1, pid

img, mask, label, pid = load_case(files[0])
print('imagem:', img.shape, img.dtype, '| min/max:', img.min(), img.max())
print('máscara:', mask.shape, '| pixels de tumor:', mask.sum())
print('rótulo (0,1,2):', label, '| paciente:', pid)

In [ ]:
import pandas as pd, os
from tqdm import tqdm

rows = []
for p in tqdm(files):
    im, mk, lb, pd_ = load_case(p)
    rows.append(dict(path=p, id=int(os.path.basename(p)[:-4]), label=lb, pid=pd_,
                     h=im.shape[0], w=im.shape[1], vmin=float(im.min()), vmax=float(im.max()),
                     mask_px=int(mk.sum()), mask_frac=float(mk.mean())))
df = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
df.to_csv(f'{BASE}/data/metadata.csv', index=False)

nomes = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}
print('Imagens por classe:'); print(df.label.map(nomes).value_counts())
print('\nPacientes no total:', df.pid.nunique())
print('Pacientes por classe:'); print(df.groupby('label').pid.nunique())
print('\nTamanhos de imagem:'); print(df[['h', 'w']].value_counts())
print('\nImagens por paciente:'); print(df.groupby('pid').size().describe())
print('\nPacientes com mais de 1 classe:', (df.groupby('pid').label.nunique() > 1).sum())
print('Máscaras vazias:', (df.mask_px == 0).sum())

In [ ]:
import matplotlib.pyplot as plt
SEED = 42
nomes = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}
fig, axs = plt.subplots(3, 3, figsize=(9, 9))
for r, lb in enumerate([0, 1, 2]):
    amostra = df[df.label == lb].sample(3, random_state=SEED)
    for c, (_, row) in enumerate(amostra.iterrows()):
        im, mk, _, _ = load_case(row.path)
        ax = axs[r, c]
        ax.imshow(im, cmap='gray', vmin=0, vmax=np.percentile(im, 99.5))
        ax.contour(mk.astype(float), levels=[0.5], colors='red', linewidths=0.8)
        ax.set_title(f'{nomes[lb]} | id {row.id}', fontsize=9)
        ax.axis('off')
plt.tight_layout()
plt.savefig(f'{BASE}/results/dataset_exemplos.png', dpi=150)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
df['classe'] = df.label.map(nomes)
df.classe.value_counts().plot.bar(ax=axs[0], title='Imagens por classe')
df.groupby('pid').size().plot.hist(bins=20, ax=axs[1], title='Imagens por paciente')
df.boxplot(column='mask_frac', by='classe', ax=axs[2])
axs[2].set_title('Fração da imagem ocupada pela máscara')
plt.suptitle('')
plt.tight_layout()
plt.savefig(f'{BASE}/results/dataset_analise.png', dpi=150)
plt.show()
print(df.groupby('classe').mask_frac.describe()[['mean', '50%', 'min', 'max']])

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd, os

SEED = 42
nomes = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}
d = df[['id', 'label', 'pid']].copy()

# divide os pacientes em 20 grupos (~5% cada), estratificados por classe
sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
d['fold'] = -1
for k, (_, idx) in enumerate(sgkf.split(d, d['label'], groups=d['pid'])):
    d.loc[d.index[idx], 'fold'] = k

test  = d[d.fold.isin([0, 1, 2])].drop(columns='fold')   # ~15%
val   = d[d.fold.isin([3, 4, 5])].drop(columns='fold')   # ~15%
train = d[d.fold >= 6].drop(columns='fold')              # ~70%

# verificação anti-vazamento: nenhum paciente em dois conjuntos
assert not set(train.pid) & set(val.pid)
assert not set(train.pid) & set(test.pid)
assert not set(val.pid) & set(test.pid)
assert len(train) + len(val) + len(test) == len(d)
print('Verificação anti-vazamento: OK\n')

splits = [('train', train), ('val', val), ('test', test)]
resumo = []
for nome, s in splits:
    r = {'conjunto': nome, 'imagens': len(s), '% imagens': round(100 * len(s) / len(d), 1), 'pacientes': s.pid.nunique()}
    for k, v in nomes.items():
        r[v] = int((s.label == k).sum())
    resumo.append(r)
print('Imagens por conjunto e classe:')
print(pd.DataFrame(resumo).to_string(index=False))
print('\nPacientes por classe em cada conjunto:')
print(pd.DataFrame({n: s.groupby('label').pid.nunique() for n, s in splits}).rename(index=nomes))

os.makedirs(f'{BASE}/data/splits', exist_ok=True)
for nome, s in splits:
    s.to_csv(f'{BASE}/data/splits/{nome}.csv', index=False)
print('\nArquivos salvos:', sorted(os.listdir(f'{BASE}/data/splits')))

In [ ]:
import numpy as np, pandas as pd, os

nomes = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}
d = df[['id', 'label', 'pid']].copy()
pac = d.groupby('pid').agg(label=('label', 'first')).reset_index()   # 1 linha por paciente

def make_split(seed):
    rng = np.random.RandomState(seed)
    test_p, val_p = [], []
    for lb in [0, 1, 2]:
        ps = pac[pac.label == lb].pid.values.copy()
        rng.shuffle(ps)
        k = int(round(0.15 * len(ps)))
        test_p += list(ps[:k]); val_p += list(ps[k:2 * k])
    return set(test_p), set(val_p)

SPLIT_SEED = None
for seed in range(500):
    t, v = make_split(seed)
    pt = d.pid.isin(t).mean() * 100
    pv = d.pid.isin(v).mean() * 100
    if abs(pt - 15) <= 1.5 and abs(pv - 15) <= 1.5:
        SPLIT_SEED = seed
        break
assert SPLIT_SEED is not None, 'nenhuma seed atendeu ao critério'
print('Seed da divisão escolhida:', SPLIT_SEED)

t, v = make_split(SPLIT_SEED)
test  = d[d.pid.isin(t)]
val   = d[d.pid.isin(v)]
train = d[~d.pid.isin(t | v)]

# verificação anti-vazamento
assert not set(train.pid) & set(val.pid)
assert not set(train.pid) & set(test.pid)
assert not set(val.pid) & set(test.pid)
assert len(train) + len(val) + len(test) == len(d)
print('Verificação anti-vazamento: OK\n')

splits = [('train', train), ('val', val), ('test', test)]
resumo = []
for nome, s in splits:
    r = {'conjunto': nome, 'imagens': len(s), '% imagens': round(100 * len(s) / len(d), 1), 'pacientes': s.pid.nunique()}
    for k, vv in nomes.items():
        r[vv] = int((s.label == k).sum())
    resumo.append(r)
print('Imagens por conjunto e classe:')
print(pd.DataFrame(resumo).to_string(index=False))
print('\nPacientes por classe em cada conjunto:')
print(pd.DataFrame({n: s.groupby('label').pid.nunique() for n, s in splits}).rename(index=nomes))

os.makedirs(f'{BASE}/data/splits', exist_ok=True)
for nome, s in splits:
    s.to_csv(f'{BASE}/data/splits/{nome}.csv', index=False)
open(f'{BASE}/data/splits/SPLIT_SEED.txt', 'w').write(str(SPLIT_SEED))
print('\nArquivos salvos:', sorted(os.listdir(f'{BASE}/data/splits')))

In [ ]:
import cv2, numpy as np, torch
from tqdm import tqdm

IMG = 224

def preprocess(img):
    p = np.percentile(img, 99.5)
    img = np.clip(img / p, 0, 1) if p > 0 else np.zeros_like(img)
    return cv2.resize(img, (IMG, IMG), interpolation=cv2.INTER_AREA)

N = len(df)
X = np.zeros((N, IMG, IMG), dtype=np.float16)
M = np.zeros((N, IMG, IMG), dtype=bool)
Y = np.zeros(N, dtype=np.int64)
for i, (_, r) in enumerate(tqdm(df.iterrows(), total=N)):
    im, mk, lb, pid = load_case(r.path)
    X[i] = preprocess(im)
    M[i] = cv2.resize(mk.astype(np.uint8), (IMG, IMG), interpolation=cv2.INTER_NEAREST).astype(bool)
    Y[i] = lb
IDS = df['id'].values

print('Máscaras vazias após redimensionar:', int((M.reshape(N, -1).sum(1) == 0).sum()))
print('X:', X.shape, X.dtype, '| min/max:', float(X.min()), float(X.max()))
np.savez_compressed(f'{BASE}/data/cache_224.npz', X=X, M=M, Y=Y, IDS=IDS)
print('Cache salvo no Drive.')

In [ ]:
import random, pandas as pd
import torchvision.transforms.v2 as T
from torch.utils.data import Dataset, DataLoader

MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
row_of = {int(i): k for k, i in enumerate(IDS)}

train_aug = T.Compose([
    T.RandomHorizontalFlip(0.5),
    T.RandomRotation(10, interpolation=T.InterpolationMode.BILINEAR),
    T.ColorJitter(brightness=0.1, contrast=0.1),
])
norm = T.Normalize(MEAN, STD)

class MRIDataset(Dataset):
    def __init__(self, csv_path, augment=False, return_mask=False):
        self.ids = pd.read_csv(csv_path)['id'].astype(int).tolist()
        self.augment, self.return_mask = augment, return_mask
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        k = row_of[self.ids[i]]
        x = torch.from_numpy(X[k].astype(np.float32)).unsqueeze(0).repeat(3, 1, 1)
        if self.augment:
            x = train_aug(x)
        x = norm(x)
        y = int(Y[k])
        if self.return_mask:
            return x, y, torch.from_numpy(M[k]), self.ids[i]
        return x, y, self.ids[i]

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

BATCH = 32
g = torch.Generator(); g.manual_seed(SEED)
S = f'{BASE}/data/splits'
train_ds = MRIDataset(f'{S}/train.csv', augment=True)
val_ds   = MRIDataset(f'{S}/val.csv')
test_ds  = MRIDataset(f'{S}/test.csv', return_mask=True)

train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_dl   = DataLoader(val_ds, BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds, BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('treino:', len(train_ds), '| validação:', len(val_ds), '| teste:', len(test_ds))

In [ ]:
import matplotlib.pyplot as plt
nomes = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}
mean = torch.tensor(MEAN).view(3, 1, 1); std = torch.tensor(STD).view(3, 1, 1)

xb, yb, ib = next(iter(train_dl))
print('batch de treino:', tuple(xb.shape), xb.dtype, '| labels:', yb[:8].tolist())
print('min/max após normalização:', round(float(xb.min()), 2), round(float(xb.max()), 2))
print('classes no batch:', torch.bincount(yb, minlength=3).tolist())

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for a, x, y in zip(axs.ravel(), xb[:8], yb[:8]):
    a.imshow((x * std + mean).clamp(0, 1)[0], cmap='gray'); a.set_title(nomes[int(y)]); a.axis('off')
plt.suptitle('Batch de treino (com augmentation)'); plt.tight_layout()
plt.savefig(f'{BASE}/results/batch_treino.png', dpi=120); plt.show()

idx = np.random.RandomState(0).choice(len(test_ds), 4, replace=False)
fig, axs = plt.subplots(1, 4, figsize=(12, 3.5))
for a, i in zip(axs, idx):
    x, y, m, _ = test_ds[int(i)]
    a.imshow((x * std + mean).clamp(0, 1)[0], cmap='gray')
    a.contour(m.numpy().astype(float), levels=[0.5], colors='red', linewidths=0.8)
    a.set_title(nomes[y]); a.axis('off')
plt.suptitle('Teste: imagem 224x224 com máscara redimensionada'); plt.tight_layout(); plt.show()

In [ ]:
import json
cfg = dict(
    img_size=IMG, canais='1 canal replicado para 3',
    intensidade='por imagem: clip no percentil 99.5 e divisao por ele',
    resize_imagem='INTER_AREA', resize_mascara='INTER_NEAREST',
    media=MEAN, desvio=STD,
    augmentation_treino='flip horizontal p=0.5; rotacao ate 10 graus; brilho e contraste 0.1',
    val_teste='sem augmentation', batch=BATCH, workers=2,
    seed=SEED, seed_da_divisao=int(SPLIT_SEED),
)
json.dump(cfg, open(f'{BASE}/results/metrics/config_preprocessamento.json', 'w'), indent=2)
print(json.dumps(cfg, indent=2))